# Zebra Crossing Detector — YOLOv8 Training on Google Colab
### Hybrid Classical + Neural Approach for FPGA Deployment
### Target Board: Edge Artix-7 (XC7A35T)

**Instructions:**
1. Upload this notebook to Google Colab: https://colab.research.google.com
2. Runtime -> Change runtime type -> **T4 GPU**
3. Run cells in order
4. Download `best.pt` to your PC when done

**Datasets Used (Combined for maximum robustness):**
- **CDSet** (Zenodo): 3,434 real images — night, rain, glare, worn markings
- **Roboflow Crosswalk**: 1,300+ images — various angles, daytime
- **Negative samples**: 300+ road/non-zebra images to prevent false positives

In [ ]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip install ultralytics roboflow albumentations gdown pyyaml -q
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# ============================================================
# CELL 2A: Download CDSet (BEST DATASET — Night/Rain/Glare)
# Source: Zenodo DOI: 10.5281/zenodo.7427088
# Contains: 3,434 images from vehicle-mounted camera
# Conditions: Night, rain, dazzling light, shadow, worn markings
# ============================================================
import os
import zipfile
import gdown
from pathlib import Path

os.makedirs('/content/dataset/cdset', exist_ok=True)

print('Downloading CDSet from Zenodo...')
print('If this fails, manually download from: https://zenodo.org/record/7427088')
print('Then upload to Colab and set CDSET_PATH below.')

# Manual alternative if auto-download fails:
# 1. Go to https://zenodo.org/record/7427088
# 2. Download the zip file
# 3. Upload to Colab using the file panel on the left
# 4. !unzip /content/CDSet.zip -d /content/dataset/cdset/

!wget -q 'https://zenodo.org/record/7427088/files/CDSet.zip' -O /content/CDSet.zip || echo 'wget failed, try manual download'

if os.path.exists('/content/CDSet.zip') and os.path.getsize('/content/CDSet.zip') > 1e6:
    print('Extracting CDSet...')
    !unzip -q /content/CDSet.zip -d /content/dataset/cdset/
    print('CDSet extracted!')
    !find /content/dataset/cdset -name '*.jpg' | wc -l
else:
    print('CDSet download failed or incomplete.')
    print('FALLBACK: We will use Roboflow dataset only (still good!)')

In [ ]:
# ============================================================
# CELL 2B: Download from Roboflow (YOLOv8-ready format)
# GO TO: https://universe.roboflow.com
# SEARCH: "crosswalk detection" or "zebra crossing"
# RECOMMENDED DATASETS (in order of quality):
#   1. Search: "Crosswalk Detection" by 'AI Research'  (~1300 images)
#   2. Search: "crosswalk"  filter by most images
# Click Export -> YOLOv8 format -> Show Download Code
# Paste the snippet below:
# ============================================================
from roboflow import Roboflow

# *** FILL THESE IN FROM ROBOFLOW EXPORT PAGE ***
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"    # From roboflow.com/settings/api
WORKSPACE        = "YOUR_WORKSPACE_NAME"  # e.g. 'crosswalk-ai'
PROJECT          = "YOUR_PROJECT_NAME"    # e.g. 'crosswalk-detection'
VERSION          = 1

if ROBOFLOW_API_KEY != "YOUR_API_KEY_HERE":
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE).project(PROJECT)
    dataset = project.version(VERSION).download("yolov8")
    ROBOFLOW_DATA_DIR = dataset.location
    print(f'Roboflow dataset at: {ROBOFLOW_DATA_DIR}')
else:
    print('Roboflow API key not set. Skipping Roboflow download.')
    print('Fill in your API key from: https://app.roboflow.com/settings/api')
    ROBOFLOW_DATA_DIR = None

In [ ]:
# ============================================================
# CELL 3: Add NEGATIVE SAMPLES (CRITICAL for zero false positives)
#
# WHY YOU NEED NEGATIVES:
#   Without them: Apple image -> model draws GREEN BOX (wrong!)
#   Without them: Plain road -> model draws GREEN BOX (wrong!)
#   With them:    Apple image -> NO BOX drawn (correct!)
#   With them:    Plain road -> NO BOX drawn (correct!)
#
# HOW YOLO HANDLES NEGATIVES:
#   - Image goes in images/train/ folder
#   - An EMPTY .txt file with same name goes in labels/train/
#   - Empty .txt = 'zero objects in this image'
#   - YOLO learns what NOT to detect
#
# NEGATIVE RATIO: ~10% of total dataset size
#   3000 positives -> add ~300 negatives
# ============================================================
import os
import urllib.request
import random
from pathlib import Path

NEGATIVE_DIR_IMAGES = Path('/content/dataset/negatives/images/train')
NEGATIVE_DIR_LABELS = Path('/content/dataset/negatives/labels/train')
NEGATIVE_DIR_IMAGES.mkdir(parents=True, exist_ok=True)
NEGATIVE_DIR_LABELS.mkdir(parents=True, exist_ok=True)

print('Setting up negative samples directory.')
print('Negative samples tell the model what NOT to detect.')
print()
print('WHAT TO PUT AS NEGATIVES (in order of importance):')
print('  1. Plain roads with NO zebra crossing    <- MOST IMPORTANT')
print('  2. Parking lot with white lines          <- High false-positive risk')
print('  3. Striped tiles, stairs, any stripes    <- Similar pattern')
print('  4. Night roads, wet roads                <- Confusing conditions')
print('  5. Random objects: apple, car, building  <- Low risk but good to include')
print()

# Download negative images from COCO (roads, parking lots, streets)
# COCO val2017 contains many street/road images — perfect negatives
print('Downloading COCO street images as negatives...')
!wget -q http://images.cocodataset.org/zips/val2017.zip -O /tmp/coco_val.zip
!unzip -q /tmp/coco_val.zip -d /tmp/coco_val/

# Filter COCO images that are roads/streets (no crosswalks)
# We pick a random sample of road-like images
import glob
import shutil

coco_images = glob.glob('/tmp/coco_val/val2017/*.jpg')
random.seed(42)
selected_negatives = random.sample(coco_images, min(300, len(coco_images)))

for i, img_path in enumerate(selected_negatives):
    dest_img = NEGATIVE_DIR_IMAGES / f'neg_{i:04d}.jpg'
    dest_lbl = NEGATIVE_DIR_LABELS / f'neg_{i:04d}.txt'
    shutil.copy(img_path, dest_img)
    # CRITICAL: Empty label file = no objects
    dest_lbl.touch()  # Creates empty file

print(f'Added {len(selected_negatives)} negative samples')
print('Each has an empty .txt label file = no objects in that image')

In [ ]:
# ============================================================
# CELL 4: Build combined dataset.yaml
# Merges CDSet + Roboflow + Negatives into one training set
# ============================================================
import yaml
import shutil
from pathlib import Path
import glob

COMBINED_DIR = Path('/content/dataset/combined')
for split in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    (COMBINED_DIR / split).mkdir(parents=True, exist_ok=True)

def copy_files(src_img_dir, src_lbl_dir, dst_img_dir, dst_lbl_dir, prefix=''):
    """Copy images and labels to combined directory with a prefix to avoid name collisions."""
    src_imgs = list(Path(src_img_dir).glob('*.jpg')) + list(Path(src_img_dir).glob('*.png'))
    count = 0
    for img_path in src_imgs:
        new_name = f"{prefix}_{img_path.stem}"
        dst_img = dst_img_dir / f"{new_name}{img_path.suffix}"
        shutil.copy(img_path, dst_img)
        
        lbl_path = Path(src_lbl_dir) / f"{img_path.stem}.txt"
        dst_lbl = dst_lbl_dir / f"{new_name}.txt"
        if lbl_path.exists():
            shutil.copy(lbl_path, dst_lbl)
        else:
            dst_lbl.touch()  # Empty label = negative
        count += 1
    return count

total_train = 0
total_val = 0

# 1. Copy negatives into combined
n = copy_files(
    NEGATIVE_DIR_IMAGES, NEGATIVE_DIR_LABELS,
    COMBINED_DIR/'images/train', COMBINED_DIR/'labels/train',
    prefix='neg'
)
print(f'Negatives added to train: {n}')
total_train += n

# 2. Copy Roboflow dataset if available
if ROBOFLOW_DATA_DIR and Path(ROBOFLOW_DATA_DIR).exists():
    for split, dst_split in [('train', 'train'), ('valid', 'val')]:
        src_img = Path(ROBOFLOW_DATA_DIR) / split / 'images'
        src_lbl = Path(ROBOFLOW_DATA_DIR) / split / 'labels'
        if src_img.exists():
            n = copy_files(
                src_img, src_lbl,
                COMBINED_DIR/f'images/{dst_split}',
                COMBINED_DIR/f'labels/{dst_split}',
                prefix=f'rf_{split}'
            )
            print(f'Roboflow {split}: {n} images')
            if dst_split == 'train': total_train += n
            else: total_val += n

# 3. Copy CDSet if available (TODO: adapt based on CDSet folder structure)
# CDSet is in YOLO format already — check and copy
cdset_train = Path('/content/dataset/cdset/train')
if cdset_train.exists():
    n = copy_files(
        cdset_train/'images', cdset_train/'labels',
        COMBINED_DIR/'images/train', COMBINED_DIR/'labels/train',
        prefix='cd_train'
    )
    print(f'CDSet train: {n} images')
    total_train += n

print(f'\nTotal train images: {total_train}')
print(f'Total val images:   {total_val}')

# Write data.yaml
data_yaml = {
    'path': str(COMBINED_DIR.absolute()),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': ['zebra_crossing']
}
COMBINED_YAML = str(COMBINED_DIR / 'data.yaml')
with open(COMBINED_YAML, 'w') as f:
    yaml.dump(data_yaml, f)

print(f'\ndata.yaml written to: {COMBINED_YAML}')
print('Dataset ready for training!')

In [ ]:
# ============================================================
# CELL 5: TRAIN THE MODEL
#
# EPOCHS EXPLANATION:
#   We set epochs=300 but patience=50.
#   This means:
#   - If mAP stops improving for 50 epochs -> training STOPS AUTOMATICALLY
#   - YOLOv8 always saves the BEST weights (not the last)
#   - So if best was epoch 120, training stops at 170, saves epoch 120 weights
#   - 300 is just a ceiling — actual training might stop at 150-200
#
# WHY NOT 50 EPOCHS?
#   - Too few = underfitting. Model doesn't learn night/rain conditions.
#   - Night images, worn markings, glare need many epochs to learn.
#
# MODEL SIZE:
#   yolov8n = nano (fastest, use if VRAM < 8GB)
#   yolov8s = small (better accuracy, use if VRAM >= 8GB)
# ============================================================
from ultralytics import YOLO
import torch

# Pick model size based on GPU VRAM
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
MODEL = 'yolov8s.pt' if vram_gb >= 8 else 'yolov8n.pt'
print(f'VRAM: {vram_gb:.1f} GB -> Using model: {MODEL}')

model = YOLO(MODEL)

results = model.train(
    data=COMBINED_YAML,

    # === EPOCHS & STOPPING ===
    epochs=300,         # Maximum ceiling — early stopping will kick in before this
    patience=50,        # Stop if no improvement for 50 epochs -> prevents overfitting

    # === IMAGE & BATCH ===
    imgsz=640,          # Training resolution (high = more accurate)
    batch=16,           # Reduce to 8 if GPU runs out of memory

    # === LEARNING RATE ===
    lr0=0.01,           # Starting LR — standard
    lrf=0.001,          # Final LR — ensures fine-tuning at end
    warmup_epochs=5,    # Warm up slowly to avoid early instability
    cos_lr=True,        # Cosine LR schedule (better than linear)

    # === REGULARIZATION (prevents overfitting) ===
    weight_decay=0.0005,
    dropout=0.1,        # Light dropout for generalization

    # === AUGMENTATIONS (handles all lighting/conditions) ===
    # These teach the model to handle:
    hsv_h=0.015,        # Hue variation (different lighting colors)
    hsv_s=0.7,          # Saturation (sunny vs cloudy vs night)
    hsv_v=0.4,          # Brightness (night, shadow, afternoon)
    degrees=10.0,       # Rotation (camera tilt)
    translate=0.1,      # Translation (different camera positions)
    scale=0.5,          # Scale jitter (far vs near crossing)
    shear=3.0,          # Shear (perspective)
    perspective=0.001,  # Perspective warp (camera angle variation)
    flipud=0.0,         # NO vertical flip (roads don't flip)
    fliplr=0.5,         # Horizontal flip OK (road is symmetric)
    mosaic=1.0,         # Mosaic (shows 4 images at once -> teaches context)
    mixup=0.1,          # MixUp (blend 2 images -> robustness)
    copy_paste=0.1,     # Copy-paste crossing into different backgrounds

    # === OUTPUT ===
    project='/content/runs',
    name='zebra_v1',
    save=True,
    save_period=20,     # Checkpoint every 20 epochs
    plots=True,
    verbose=True,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    workers=4,
    seed=42,            # Reproducibility for IEEE paper

    # === CONFIDENCE ===
    conf=0.001,         # Low threshold during training (default)
    iou=0.7,            # NMS IoU threshold
)

print('\nTraining Complete!')
print(f'Best mAP50:    {results.results_dict.get("metrics/mAP50(B)", "N/A")}')
print(f'Best mAP50-95: {results.results_dict.get("metrics/mAP50-95(B)", "N/A")}')
print(f'Stopped at epoch: check /content/runs/zebra_v1/results.csv')

In [ ]:
# ============================================================
# CELL 6: View Training Graphs
# These help you understand if training was good
# ============================================================
from IPython.display import Image as IPImage, display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

plots_dir = '/content/runs/zebra_v1/'

# Show training results plot
results_png = plots_dir + 'results.png'
if os.path.exists(results_png):
    print('=== TRAINING CURVES ===')
    print('Look for:')
    print('  - val/box_loss, val/cls_loss: Should DECREASE and FLATTEN (not rise!)')
    print('  - metrics/mAP50: Should INCREASE and FLATTEN')
    print('  - If val_loss rises while train_loss drops = OVERFITTING')
    display(IPImage(results_png, width=900))

# Show confusion matrix
cm_png = plots_dir + 'confusion_matrix_normalized.png'
if os.path.exists(cm_png):
    print('\n=== CONFUSION MATRIX ===')
    print('Background column = false positives on non-zebra images')
    print('zebra_crossing column = true detections')
    display(IPImage(cm_png, width=500))

In [ ]:
# ============================================================
# CELL 7: Validate on test set
# ============================================================
from ultralytics import YOLO

best_model = YOLO('/content/runs/zebra_v1/weights/best.pt')

val_results = best_model.val(
    data=COMBINED_YAML,
    imgsz=640,
    conf=0.3,     # Confidence threshold for deployment
    iou=0.5,
    device='cuda'
)

print('\n========== VALIDATION RESULTS ==========')
print(f'mAP@50:        {val_results.box.map50:.4f}  (target: >0.85)')
print(f'mAP@50-95:     {val_results.box.map:.4f}  (target: >0.70)')
print(f'Precision:     {val_results.box.mp:.4f}  (= how often detection is correct)')
print(f'Recall:        {val_results.box.mr:.4f}  (= how often crossing is NOT missed)')
print('========================================')
print()
print('WHAT THESE MEAN:')
print('  Precision = TP / (TP + FP)  <- Low = too many false positives (draws boxes on non-zebra)')
print('  Recall    = TP / (TP + FN)  <- Low = misses real zebra crossings')
print('  For safety-critical app: Recall > Precision (better to detect more)')

In [ ]:
# ============================================================
# CELL 8: Test on custom images (your Google test images)
# ============================================================
from ultralytics import YOLO
from IPython.display import Image as IPImage, display
import cv2
import numpy as np
import os

best_model = YOLO('/content/runs/zebra_v1/weights/best.pt')

# Test with your own images!
# Option A: Upload images and test
# from google.colab import files
# uploaded = files.upload()
# test_paths = list(uploaded.keys())

# Option B: Use URLs
import urllib.request
test_images = [
    # Paste any image URL here for testing:
    # ('zebra_test.jpg', 'https://example.com/zebra.jpg'),  
    # ('road_test.jpg', 'https://example.com/road.jpg'),    
]

for fname, url in test_images:
    try:
        urllib.request.urlretrieve(url, f'/tmp/{fname}')
        results = best_model(f'/tmp/{fname}', conf=0.3)
        img_annotated = results[0].plot()
        cv2.imwrite(f'/tmp/result_{fname}', img_annotated)
        print(f'\nImage: {fname}')
        if len(results[0].boxes) > 0:
            for box in results[0].boxes:
                x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
                print(f'  DETECTED: x1={x1} y1={y1} x2={x2} y2={y2} conf={box.conf[0]:.3f}')
        else:
            print('  No zebra crossing detected.')
        display(IPImage(f'/tmp/result_{fname}'))
    except Exception as e:
        print(f'Error on {fname}: {e}')

print()
print('SHOW THESE RESULTS TO YOUR SUPERVISOR/COLLABORATOR FOR APPROVAL')
print('Before proceeding to FPGA integration!')

In [ ]:
# ============================================================
# CELL 9: Download model to your PC
# ============================================================
from google.colab import files
import shutil

# Copy best model
shutil.copy('/content/runs/zebra_v1/weights/best.pt', '/content/zebra_best.pt')

print('IMPORTANT: Save best.pt to this path on your PC:')
print(r'  C:\Users\Cambidge Qwality\.gemini\antigravity\scratch\zebra-fpga-detector\runs\detect\train\weights\best.pt')
print()

files.download('/content/zebra_best.pt')

# Download training plots for IEEE paper figures
!zip -r /content/training_plots.zip /content/runs/zebra_v1/*.png /content/runs/zebra_v1/results.csv
files.download('/content/training_plots.zip')

print('Done! Files downloading...')

In [ ]:
# ============================================================
# CELL 10: Export to ONNX (for future FPGA quantization)
# ============================================================
from ultralytics import YOLO

best_model = YOLO('/content/runs/zebra_v1/weights/best.pt')

# Export to ONNX at 320x320 (FPGA inference size)
best_model.export(
    format='onnx',
    imgsz=320,
    opset=12,
    simplify=True,
    dynamic=False,
    half=False,   # Keep float32 for now; quantize separately
)
print('ONNX model exported at 320x320 resolution')
print('This will be used for FPGA quantization in Stage 2')

from google.colab import files
files.download('/content/runs/zebra_v1/weights/best.onnx')